In [7]:
!pip install -q dbt-duckdb

In [8]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id   = vl.workspace_id
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_name = vl.lakehouse_name
    dbt_path       = vl.dbt_path
    metadata_path  = vl.metadata_path
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get('id')
    in_fabric      = True
except ModuleNotFoundError:
    import yaml
    from pathlib import Path
    _p = Path.cwd()
    _root = next(
        (parent for parent in [_p, *_p.parents] if (parent / "deploy_config.yml").exists()),
        None,
    )
    if _root is None:
        raise FileNotFoundError("deploy_config.yml not found in cwd or any parent")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws_id"]
    lakehouse_id   = _cfg["lakehouse_id"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = str(_root / "dbt")
    metadata_path  = _cfg["metadata_path"]
    in_fabric      = False
os.environ['ROOT_PATH']      = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}'
os.environ['download_limit'] = download_limit
os.environ['process_limit']  = process_limit

In [9]:
"""
OneLake helpers — metadata.db lease + dbt project download.

The lease protects DuckLake's SQLite catalog from concurrent writers.
The download grabs the dbt project so we never have to mount the lakehouse.
See README "Concurrency safety" section for the lock details.
"""
import os, sqlite3, tempfile
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone


def _onelake_auth():
    """Return (azure-sdk credential, holder-label). Fabric token if available, else `az login`."""
    from azure.core.credentials import AccessToken, TokenCredential
    try:
        import notebookutils
        token = notebookutils.credentials.getToken("storage")
        class _StaticToken(TokenCredential):
            def get_token(self, *_, **__): return AccessToken(token, 9999999999)
        return _StaticToken(), "fabric-notebook"
    except ModuleNotFoundError:
        from azure.identity import DefaultAzureCredential
        return DefaultAzureCredential(), "local-dev"


def _onelake_rel(path, lakehouse_id):
    """/lakehouse/default/Files/foo -> {lakehouse_id}/Files/foo"""
    marker = "/Files/"
    idx = path.find(marker)
    if idx < 0:
        raise ValueError(f"path missing '/Files/' segment: {path}")
    return f"{lakehouse_id}/Files/{path[idx + len(marker):]}"


def download_dbt_from_onelake(dbt_path, workspace_id, lakehouse_id, local_dir=None):
    """
    Mirror the dbt project folder from OneLake to a local tmp dir and return
    the local path. Lets dbt run without any lakehouse mount.
    """
    from azure.storage.filedatalake import DataLakeServiceClient

    if local_dir is None:
        local_dir = os.path.join(tempfile.gettempdir(), "dbt")
    credential, _ = _onelake_auth()
    rel = _onelake_rel(dbt_path, lakehouse_id)
    fs = (
        DataLakeServiceClient(
            "https://onelake.dfs.fabric.microsoft.com",
            credential=credential,
        )
        .get_file_system_client(workspace_id)
    )
    for p in fs.get_paths(path=rel, recursive=True):
        if p.is_directory:
            continue
        suffix = os.path.relpath(p.name, rel)
        target = os.path.join(local_dir, suffix)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with open(target, "wb") as fh:
            fh.write(fs.get_file_client(p.name).download_file().readall())
    return local_dir


@contextmanager
def onelake_metadata_lock(metadata_path, workspace_id, lakehouse_id,
                          local_path=None, stale_hours=12):
    """
    Hold an exclusive lease on metadata.db on OneLake while dbt runs.

    Flow:
      1. If the OneLake file is missing, seed it with a valid empty SQLite db.
      2. Acquire an infinite lease; auto-break if the prior lease is older
         than `stale_hours` (self-heal from a crashed run).
      3. Download to `local_path`, yield it.
      4. On exit: upload modifications back under the lease, then release.

    A second run hitting the same file fails fast with LeaseAlreadyPresent.
    """
    from azure.storage.filedatalake import DataLakeServiceClient, DataLakeLeaseClient
    from azure.core.exceptions import ResourceNotFoundError, HttpResponseError

    credential, holder = _onelake_auth()

    tmp = tempfile.gettempdir()
    if local_path is None:
        local_path = os.path.join(tmp, "metadata.db")

    file = (
        DataLakeServiceClient(
            "https://onelake.dfs.fabric.microsoft.com",
            credential=credential,
        )
        .get_file_system_client(workspace_id)
        .get_file_client(_onelake_rel(metadata_path, lakehouse_id))
    )

    # First-run seed: ADLS won't lease an unflushed zero-byte blob, and
    # DuckLake can't ATTACH an empty file. CREATE+DROP forces SQLite to
    # write a real header, giving us a leasable, attach-able placeholder.
    try:
        file.get_file_properties()
    except ResourceNotFoundError:
        seed = os.path.join(tmp, "_metadata_seed.db")
        c = sqlite3.connect(seed)
        c.execute("CREATE TABLE _bootstrap (x)")
        c.execute("DROP TABLE _bootstrap")
        c.commit(); c.close()
        with open(seed, "rb") as fh:
            file.upload_data(fh.read(), overwrite=True)

    # Acquire — auto-heal a stale lease (>stale_hours) from a crashed run.
    try:
        lease = file.acquire_lease(lease_duration=-1)
    except HttpResponseError as e:
        if e.error_code != "LeaseAlreadyPresent":
            raise
        stamped = (file.get_file_properties().metadata or {}).get("acquired_at")
        is_stale = True
        if stamped:
            try:
                age = datetime.now(timezone.utc) - datetime.fromisoformat(stamped)
                is_stale = age > timedelta(hours=stale_hours)
            except ValueError:
                pass
        if not is_stale:
            raise RuntimeError(
                f"metadata.db is locked by an active writer (acquired_at={stamped}). "
                f"Refusing to run. Stale leases auto-break after {stale_hours}h."
            )
        DataLakeLeaseClient(file).break_lease(lease_break_period=0)
        lease = file.acquire_lease(lease_duration=-1)

    try:
        file.set_metadata(
            {"acquired_at": datetime.now(timezone.utc).isoformat(),
             "holder":      holder},
            lease=lease.id,
        )
        with open(local_path, "wb") as fh:
            fh.write(file.download_file(lease=lease.id).readall())
        try:
            yield local_path
        finally:
            with open(local_path, "rb") as fh:
                file.upload_data(fh.read(), overwrite=True, lease=lease.id)
    finally:
        lease.release()

In [10]:
from dbt.cli.main import dbtRunner

def dbt_run():
    os.chdir(dbt_path)
    dbtRunner().invoke(["run", "--target", "prod", "--profiles-dir", "."])

def dbt_test():
    os.chdir(dbt_path)
    dbtRunner().invoke(["test", "--target", "prod", "--profiles-dir", "."])

In [11]:
if in_fabric:
    dbt_path = download_dbt_from_onelake(dbt_path, workspace_id, lakehouse_id)

# Lock held only for the write phase. Tests are read-only, no lease needed —
# they run against the local snapshot after the upload+release.
with onelake_metadata_lock(metadata_path, workspace_id, lakehouse_id) as local_db:
    os.environ['METADATA_LOCAL_PATH'] = local_db
    dbt_run()

dbt_test()

06:01:57  Running with dbt=1.11.8
06:01:58  Registered adapter: duckdb=1.10.1
06:02:03  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on 'fct_price' in
package 'aemo_electricity' (models\marts\schema.yml). Arguments to generic tests
should be nested under the `arguments` property.
06:02:03  Found 8 models, 38 data tests, 7 operations, 479 macros
06:02:03  
06:02:03  Concurrency: 1 threads (target='prod')
06:02:03  
06:02:13  1 of 3 START hook: aemo_electricity.on-run-start.0 ............................. [RUN]
06:02:13  1 of 3 OK hook: aemo_electricity.on-run-start.0 ................................ [OK in 0.03s]
06:02:13  2 of 3 START hook: aemo_electricity.on-run-start.1 ............................. [RUN]
06:02:13  2 of 3 OK hook: aemo_electricity.on-run-start.1 ................................ [OK in 0.04s]
06:02:13  3 of 3 START hook: aemo_electricity.on-run-start.2 ................

In [12]:
# Escape hatch: manually break the metadata.db lease.
# Uncomment and run this cell to force-release a lease left behind by a
# crashed run (only needed before the 12h stale-auto-break kicks in).
#
# from azure.storage.filedatalake import DataLakeServiceClient, DataLakeLeaseClient
# credential, _ = _onelake_auth()
# file = (
#     DataLakeServiceClient(
#         "https://onelake.dfs.fabric.microsoft.com",
#         credential=credential,
#     )
#     .get_file_system_client(workspace_id)
#     .get_file_client(_onelake_rel(metadata_path, lakehouse_id))
# )
# DataLakeLeaseClient(file).break_lease(lease_break_period=0)
# print("lease broken")